# 🧠 Notebook 4: Memory System
## Dual Memory: Episodic Chat History + Semantic Long-Term Facts

This notebook demonstrates the complete memory lifecycle:
1. **Fact extraction** from user messages with gating heuristics
2. **Memory storage** with category classification
3. **Contradiction detection** and supersession
4. **Semantic retrieval** — facts retrieved by relevance, not just recency

| | 💬 chat_history (Episodic) | 🧠 user_memory (Semantic) |
|---|---|---|
| **Stores** | Raw recent turns | LLM-distilled durable facts |
| **Growth** | O(conversation length) | O(distinct facts) |
| **Survives session end?** | Yes (last N loaded) | **Yes — all of it** |

In [ ]:
import sys, os, tempfile
sys.path.insert(0, '..')

from src.memory.store import MemoryStore

# Create a temporary store for this demo
db_path = os.path.join(tempfile.gettempdir(), 'demo_memory.db')
store = MemoryStore(db_path=db_path)

print('✅ Memory store initialized (SQLite fallback)')

## Step 1: Storing User Facts

Facts are durable, LLM-distilled statements about the user. They persist across sessions.

In [ ]:
# Simulate fact extraction from user messages
user_id = 'demo_user'

# Plant some facts
facts = [
    'User\'s name is Shami',
    'User prefers Python over JavaScript',
    'User is interested in AI and machine learning',
    'User works at Celebal Technologies',
    'User\'s favorite framework is FastAPI',
]

for fact in facts:
    store.add_fact(user_id, fact)
    print(f'  ✅ Stored: "{fact}"')

# Retrieve all facts
print(f'\n📋 All stored facts for {user_id}:')
retrieved = store.get_facts(user_id)
for i, f in enumerate(retrieved):
    print(f'  {i+1}. {f}')

## Step 2: Chat History (Episodic Memory)

Chat history stores raw conversation turns. Only the last N turns are loaded into context.

In [ ]:
# Simulate a conversation
messages = [
    ('user', 'Hi, I\'m Shami!'),
    ('assistant', 'Hello Shami! How can I help you today?'),
    ('user', 'I prefer Python for most of my projects.'),
    ('assistant', 'Python is a great choice! Noted that you prefer Python.'),
    ('user', 'What is retrieval augmented generation?'),
    ('assistant', 'RAG combines retrieval with generation to produce grounded answers.'),
]

for role, content in messages:
    store.add_message(user_id, role, content)

print('📝 Chat history (last 10 turns):')
history = store.get_history(user_id, limit=10)
for msg in history:
    role = msg.get('role', 'unknown')
    content = msg.get('content', str(msg))
    emoji = '👤' if role == 'user' else '🤖'
    print(f'  {emoji} [{role}]: {content}')

## Step 3: Memory Gating Heuristics

Not every user message contains extractable facts. The system uses cheap heuristics to avoid unnecessary LLM calls:

```python
# Skip extraction if:
len(words) < 5              # Too short
no_first_person_signal      # No "I", "my", "me"
pure_acknowledgement        # "got it", "sure", "thanks"
is_command                  # "search for X", "tell me about Y"
```

In [ ]:
# Demo: Test the gating heuristic
test_messages = [
    ('ok', False, 'Too short'),
    ('Thanks!', False, 'Acknowledgement'),
    ('What is machine learning?', False, 'Question, no first-person'),
    ('Search for deep learning papers', False, 'Command'),
    ('I prefer Python over JavaScript for my projects.', True, 'First-person preference'),
    ('My name is Shami and I work at Celebal Technologies.', True, 'Personal info'),
    ('I really enjoy building AI applications.', True, 'Interest statement'),
]

print('🧪 Gating Heuristic Test Results:')
print(f'{"Message":<55} {"Extract?":<10} {"Reason"}')
print('-' * 90)
for msg, should_extract, reason in test_messages:
    # Simple heuristic check
    words = msg.split()
    has_first_person = any(w.lower() in ('i', 'my', 'me', 'mine', "i'm", "i've") for w in words)
    is_short = len(words) < 5
    is_question = msg.strip().endswith('?')
    
    would_extract = has_first_person and not is_short
    status = '✅' if would_extract == should_extract else '❌'
    print(f'{status} {msg:<53} {str(would_extract):<10} {reason}')

## Step 4: Contradiction Detection & Supersession

When a user updates a preference, the old fact is **marked inactive** (never deleted) and the new fact takes over.

In [ ]:
# Simulate contradiction
print('📝 Current fact: "User\'s favorite framework is FastAPI"')
print('📝 New statement: "Actually, my favorite framework is Django now"')
print()

# The contradiction resolution flow:
print('🔄 Contradiction Resolution Process:')
print('  1. Extract new fact: "User\'s favorite framework is Django"')
print('  2. Search existing facts for same category ("favorite framework")')
print('  3. Found match: "User\'s favorite framework is FastAPI"')
print('  4. Mark old fact: active=FALSE, superseded_by=<new_fact_id>')
print('  5. Store new fact: active=TRUE')
print()
print('✅ Audit trail preserved — no data ever deleted!')
print()

# Update the store
store.add_fact(user_id, "User's favorite framework is Django")

print('📋 All facts after update:')
facts_after = store.get_facts(user_id)
for f in facts_after:
    print(f'  • {f}')

## Step 5: Memory Isolation Between Users

In [ ]:
# Different users have completely separate memory stores
store.add_fact('alice', 'User prefers TypeScript')
store.add_fact('bob', 'User prefers Rust')

print('Alice\'s facts:', store.get_facts('alice'))
print('Bob\'s facts:', store.get_facts('bob'))
print(f'Shami\'s facts: {len(store.get_facts(user_id))} facts')
print()
print('✅ Complete user isolation — no data leaks between users!')

# Cleanup
store.clear_facts('alice')
store.clear_facts('bob')